# Model Inference — Best Model Submission
Loads best registered model from MLflow Model Registry and generates test predictions.

In [1]:
!pip install dagshub mlflow scikit-learn xgboost pandas numpy -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/

In [2]:
import os, warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
warnings.filterwarnings('ignore')

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['MLFLOW_TRACKING_USERNAME'] = 'dgrig23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = secrets.get_secret('DAGSHUB_TOKEN')

REPO = 'dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning'
mlflow.set_tracking_uri(f'https://dagshub.com/{REPO}.mlflow')
client = mlflow.tracking.MlflowClient()
print('Connected to MLflow tracking server.')


Connected to MLflow tracking server.


## 1. Compare All Registered Models

In [3]:
METRIC_PRIORITY = ['final_oof_auc', 'val_auc', 'final_cv_auc',
                   'best_cv_score', 'best_cv_auc', 'test_auc']

def get_best_auc(metrics: dict) -> float:
    """Return the first matching AUC metric in priority order."""
    for m in METRIC_PRIORITY:
        if m in metrics and metrics[m] is not None:
            return metrics[m]
    return None

registered_models = client.search_registered_models()

results = []
for rm in registered_models:
    try:
        versions = client.search_model_versions(f"name='{rm.name}'")
        if not versions: continue
        latest = sorted(versions, key=lambda v: int(v.version))[-1]
        run = client.get_run(latest.run_id)
        metrics = run.data.metrics
        auc = get_best_auc(metrics)
        metric_used = next((m for m in METRIC_PRIORITY if m in metrics), 'none')
        results.append({
            'model':        rm.name,
            'version':      latest.version,
            'run_id':       latest.run_id,
            'auc':          auc,
            'metric_used':  metric_used,
        })
    except Exception as e:
        print(f'  Warning — {rm.name}: {e}')

results_df = (pd.DataFrame(results)
              .sort_values('auc', ascending=False, na_position='last')
              .reset_index(drop=True))

print(results_df[['model','version','auc','metric_used']].to_string(index=False))

valid = results_df.dropna(subset=['auc'])
if valid.empty:
    raise RuntimeError('No models with a valid AUC found in the registry!')

best_row = valid.iloc[0]
best_model_name = best_row['model']
best_version    = best_row['version']
best_auc        = best_row['auc']
print(f'\n→ Best model: {best_model_name}  version={best_version}  AUC={best_auc:.5f}')


                            model version      auc   metric_used
      RandomForest_FraudDetection       1 0.999660       val_auc
           XGBoost_FraudDetection       2 0.970200 final_oof_auc
          AdaBoost_FraudDetection       2 0.864510  final_cv_auc
      DecisionTree_FraudDetection       2 0.856070  final_cv_auc
LogisticRegression_FraudDetection       3 0.814333 best_cv_score

→ Best model: RandomForest_FraudDetection  version=1  AUC=0.99966


## 2. Load Best Model from Registry

In [4]:
model_uri = f'models:/{best_model_name}/{best_version}'
print(f'Loading: {model_uri} ...')

pipeline = mlflow.sklearn.load_model(model_uri)
print('Loaded:', type(pipeline))
if hasattr(pipeline, 'steps'):
    print('Pipeline steps:', [name for name, _ in pipeline.steps])


Loading: models:/RandomForest_FraudDetection/1 ...


Loaded: <class 'sklearn.pipeline.Pipeline'>
Pipeline steps: ['preprocessor', 'selector', 'classifier']


## 3. Load Test Data

In [5]:
BASE = '/kaggle/input/competitions/ieee-fraud-detection/'

test_trx = pd.read_csv(BASE + 'test_transaction.csv')
test_idn = pd.read_csv(BASE + 'test_identity.csv')
test_idn.columns = test_idn.columns.str.replace('-', '_')
test_trx.columns = test_trx.columns.str.replace('-', '_')

test     = test_trx.merge(test_idn, on='TransactionID', how='left')
test_ids = test['TransactionID'].copy()

print(f'Test shape: {test.shape}  |  IDs: {len(test_ids)}')


Test shape: (506691, 433)  |  IDs: 506691


## 4. Generate & Save Predictions

In [6]:
y_proba = pipeline.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud':       y_proba
})

assert submission.shape[0] == len(test_ids),          'Row count mismatch!'
assert submission['isFraud'].between(0, 1).all(),      'Probabilities outside [0,1]!'
assert submission['TransactionID'].nunique() == len(test_ids), 'Duplicate IDs!'

output_path = '/kaggle/working/submission.csv'
submission.to_csv(output_path, index=False)

print(f'Saved {len(submission)} rows to {output_path}')
print(f'Prediction stats: min={y_proba.min():.4f} | max={y_proba.max():.4f}'
      f' | mean={y_proba.mean():.4f} | fraud_predicted={(y_proba>0.5).mean()*100:.2f}%')
print(submission.head(10))


Saved 506691 rows to /kaggle/working/submission.csv
Prediction stats: min=0.0000 | max=0.9997 | mean=0.0796 | fraud_predicted=1.80%
   TransactionID   isFraud
0        3663549  0.027141
1        3663550  0.030234
2        3663551  0.035645
3        3663552  0.034124
4        3663553  0.055234
5        3663554  0.025355
6        3663555  0.143315
7        3663556  0.153426
8        3663557  0.014893
9        3663558  0.061845
